# **Decision Tree Algorithm** 

Prerequisites: Probability primer (log, information-theoretic intuition).

## **1. Introduction** 

A **Decision Tree** is a supervised learning algorithm used for **classification** and **regression** tasks. It recursively splits the dataset into smaller subsets based on feature values, creating a tree-like structure. 
- **Classification Decision Trees**: Predict categorical outcomes (e.g., spam or not spam). 
- **Regression Decision Trees**: Predict continuous values (e.g., house prices). 

##
---

## **2. Key Mathematical Concepts** 

### **2.1 Splitting Criteria** 
Decision trees split nodes using specific mathematical criteria to determine the "best" split. Common criteria include: 


#### 1. **Gini Impurity (Classification)** 
Measures the likelihood of an incorrect classification at a node: 
$$ 
    Gini = 1-\sum_{k=1}^K p_k^2 = \sum_{k}p_k(1-p_k) 
$$ 
- \(p_k\): Proportion of instances belonging to class \(k\). 
- \(K\): Total number of classes.

— the probability of misclassifying a randomly-drawn point if you labeled
it randomly according to the node's class distribution. Faster to compute
than entropy (no logarithm) — the reason CART (sklearn's default) uses
Gini rather than entropy for splits, though results are usually similar.

#### 2. **Entropy (Classification) - derivation from information theory** 
Entropy measures uncertainty or impurity in a node: 
$$ 
    Entropy = - \sum_{i=1}^{C} p_k \cdot \log_2(p_k) 
$$ 

- Lower entropy indicates purer splits.

Intuition: $-\log_2 p_k$ is the "surprise"/information content of
observing class $k$ (rarer events are more surprising); entropy is the
**expected surprise** — maximized when classes are equally likely (most
uncertain), zero when the node is pure (one class, $p_k=1$).

#### 3. **Information Gain (Classification)** 
The reduction in entropy due to splitting: 
$$ 
    Information \, Gain = Entropy(parent) - \sum_{k=1}^{K} \frac{|D_k|}{|D|} \cdot Entropy(D_k) 
$$ 
- \( D_k \): Subset of data after the split. 
- \( |D_k| \): Number of samples in \(D_k\). 
- \( |D| \): Total samples in the parent node.

#### 4. **Mean Squared Error (MSE) (Regression)** 
Measures the variance within a node for regression tasks: 
$$ 
    MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y})^2 
$$ 

Where:
- \( y_i \): True value. 
- \( $\hat{y}$ \): Predicted value. 
- \( n \): Number of samples.


#### 5. **Worked Numerical Example - full split selection by hand**

8 samples, feature "Weather" ∈ {Sunny, Rainy}, feature "Weekend" ∈ {Yes,No},
label "Played" ∈ {Yes,No}:

| Weather | Weekend | Played |
|---|---|---|
| Sunny | Yes | Yes |
| Sunny | Yes | Yes |
| Sunny | No | No |
| Rainy | Yes | Yes |
| Rainy | No | No |
| Rainy | No | No |
| Sunny | No | Yes |
| Rainy | Yes | No |

Root: 5 No, 3 Yes → wait, recount: Yes count = rows 1,2,4,7 = 4; No count
= rows 3,5,6,8 = 4. $p_{yes}=4/8=0.5$, $p_{no}=0.5$.
$$H(\text{root}) = -0.5\log_2 0.5 - 0.5\log_2 0.5 = -0.5(-1)-0.5(-1)=1.0 \text{ bit (maximum uncertainty)}$$

**Split on Weather**: Sunny={rows 1,2,3,7}=3 Yes,1 No; Rainy={rows 4,5,6,8}=1 Yes,3 No.
$$H(\text{Sunny}) = -\tfrac34\log_2\tfrac34-\tfrac14\log_2\tfrac14 = -0.75(-0.415)-0.25(-2.0)=0.311+0.5=0.811$$
By symmetry $H(\text{Rainy})=0.811$ (same 3/1 split, other class).
$$IG(\text{Weather}) = 1.0 - \left[\tfrac48(0.811)+\tfrac48(0.811)\right] = 1.0-0.811=0.189$$

**Split on Weekend**: Yes={rows1,2,4,8}=3 Yes,1 No; No={rows3,5,6,7}=1 Yes,3 No.
Identical counts to the Weather split by coincidence in this toy example →
$H(\text{Yes})=H(\text{No})=0.811$, $IG(\text{Weekend})=0.189$ too — a tie,
illustrating that real datasets need a tie-breaking rule (e.g. first
feature encountered, or gain ratio to correct for features with more
categories — worth mentioning C4.5's gain-ratio fix here explicitly).

In [1]:
import numpy as np

def entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))

root = ["Y","Y","N","Y","N","N","Y","N"]
sunny = ["Y","Y","N","Y"]   # rows 1,2,3,7
rainy = ["Y","N","N","N"]   # rows 4,5,6,8

H_root = entropy(root)
H_sunny = entropy(sunny)
H_rainy = entropy(rainy)
IG = H_root - (len(sunny)/len(root))*H_sunny - (len(rainy)/len(root))*H_rainy
print(H_root, H_sunny, H_rainy, IG)   # 1.0  0.8113  0.8113  0.1887 -- matches hand calc

1.0 0.8112781244591328 0.8112781244591328 0.1887218755408671


### **2.2 Tree Pruning**

- Pruning reduces overfitting by limiting the size of the tree.
  - **Pre-pruning**: Stop growing the tree early (e.g., limit depth, minimum samples per split).
  - **Post-pruning**: Remove branches after the tree is fully grown.

### **2.3 Hyperparameters**

1. **max_depth**: Maximum depth of the tree.
2. **min_samples_split**: Minimum samples required to split a node.
3. **criterion**: The function used to measure the quality of a split (`gini` or `entropy`).
4. **max_features**: Number of features to consider when splitting.

##
---

## **3. Decision Tree Structure** 

1. **Root Node**: The starting point of the tree where data splits. 
2. **Internal Nodes**: Intermediate decision points. 
3. **Leaf Nodes**: Terminal nodes providing the output. 

Example: 
For a dataset predicting whether to play tennis:
- Root Node: Weather
- Internal Nodes: Outlook, Humidity
- Leaf Nodes: Play (Yes/No)


![Decision Tree Example.png](../../images/dta_example.png)

But there are so many ways to create the decision trees for one problem. 
<br>
<img src = "../../images/dta_ways.png" width = 70%>

### Find the correct way or Decision Tree

[Read More](#21-splitting-criteria)

Ex : By using **Entropy**

![Decision Tree Entropy Calculation](../../images/dta_entropy_calc.png)
<br><br>
- Lower entropy tree is used to develop the model

##
---

## **4. Implementation** 

### Classification Example

Dataset: Iris Classification

This example predicts the type of iris flower using a Decision Tree Classifier.

```python
# Import Libraries
from sklearn.datasets import load_iris, fetch_california_housing 
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, mean_squared_error 
import matplotlib.pyplot as plt

# Load Iris Dataset
iris = load_iris() 
X, y = iris.data, iris.target 

# Split into Training and Test Sets 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42) 

# Initialize Decision Tree Classifier 
clf = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42) 

# Train the Model 
clf.fit(X_train, y_train)

# Predict and Evaluate 
y_pred = clf.predict(X_test) 
accuracy = accuracy_score(y_test, y_pred) 
print(f"Accuracy: {accuracy:.2f}") 

# Visualize the Decision Tree 
plt.figure(figsize=(12, 8)) 
plot_tree(clf, feature_names=iris.feature_names, class_names=iris.target_names, filled=True) 
plt.show() 
```

[Implementation Of Decision Tree Algorithm](08%20-%20Implement%20Decision%20Tree%20Algorithm.ipynb#decision-tree-algorithm-for-classification-problem)
###
---

### Regression Example

Dataset: California Housing Prices

This example predicts house prices using a Decision Tree Regressor.

```python
# Load California Housing Dataset 
data = fetch_california_housing() 
X, y = data.data, data.target 

# Split into Training and Test Sets 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42) 

# Initialize Decision Tree Regressor 
regressor = DecisionTreeRegressor(max_depth=5, random_state=42) 

# Train the Model 
regressor.fit(X_train, y_train) 

# Predict and Evaluate 
y_pred = regressor.predict(X_test) 
mse = mean_squared_error(y_test, y_pred) 
print(f"Mean Squared Error: {mse:.2f}") 
```

[Implementation Of Decision Tree Algorithm](08%20-%20Implement%20Decision%20Tree%20Algorithm.ipynb#decision-tree-algorithm-for-regression-problem)
###
---

## **5. Visualization and Export**

- ### Plotting the Tree

```python
from sklearn.tree import plot_tree 

# Visualize the Decision Tree 
plt.figure(figsize=(12, 8)) 
plot_tree(clf, feature_names=iris.feature_names, class_names=iris.target_names, filled=True) 
plt.show() 
```

- ### Exporting the Tree as Text

```python
from sklearn.tree import export_text 

# Export the Tree as Text 
tree_rules = export_text(clf, feature_names=iris.feature_names) 
print(tree_rules) 
```

##
---

## **6. Common Issues and Solutions**

- ### Overfitting

- Trees can grow excessively, leading to overfitting.

- Solution: Prune the tree or use hyperparameter constraints: 
  - Use pruning

  - Limit max_depth.

  - Set min_samples_split.

- ### Imbalanced Data

- Trees may be biased toward majority classes.

- Solution: 

  - Use class weights (class_weight='balanced').

  - Resample the data (e.g., SMOTE).

##
---

## **7. Mathematical Formulation of Splits**

### Classification Example

- #### Gini Impurity Calculation:

Consider a node with:

- Class 1: 4 samples

- Class 2: 6 samples

$$    
    Gini = 1 - \left(\frac{4}{10}\right)^2 - \left(\frac{6}{10}\right)^2 = 1 - 0.16 - 0.36 = 0.48 
$$

### Regression Example

- #### Mean Squared Error Calculation:

For a node with predictions:

- Actual: y=[3,5,4]

- Predicted Mean: $\hat{y} = 4$

$$
    MSE = \frac{1}{3} \left((3 - 4)^2 + (5 - 4)^2 + (4 - 4)^2\right) = \frac{1}{3} (1 + 1 + 0) = 0.67 
$$

##
---

## 8. Advantages and Disadvantages

- ### Advantages

- Easy to interpret and visualize.

- Handles both numerical and categorical data.

- Requires minimal data preprocessing.

- ### Disadvantages

- Prone to overfitting (requires pruning or setting constraints).

- Sensitive to small variations in data (can lead to different splits).

- Not optimal for very large datasets.

- Requires careful tuning of hyperparameters.

##
---

## 9. Extensions

1. Random Forest:

  - Combines multiple decision trees to improve accuracy.

```python
from sklearn.ensemble import RandomForestClassifier 
rf = RandomForestClassifier(n_estimators=100, random_state=42) 
rf.fit(X_train, y_train) 
```

2. Gradient Boosting:

  - Builds trees sequentially to correct errors of previous trees.

```python
from sklearn.ensemble import GradientBoostingClassifier 
gb = GradientBoostingClassifier(random_state=42) 
gb.fit(X_train, y_train) 
```

3. XGBoost:

  - Optimized version of Gradient Boosting.

```python
from xgboost import XGBClassifier 
xgb = XGBClassifier(random_state=42) 
xgb.fit(X_train, y_train) 
```

## 10. Summary

|Parameter         | Description                             |
|------------------|-----------------------------------------|
|max_depth         |Maximum depth of the tree.               |
|min_samples_split |Minimum samples required to split a node.|
|criterion         |Splitting criteria (`gini, entropy`).    |
|splitter          |Splitter method (`best, random`).        |

##
---

## 11. Conclusion

Decision Trees are a powerful, interpretable model for supervised learning tasks. Their effectiveness can be improved through pruning and ensemble techniques like Random Forests or Boosting.



### Key Takeaways

- Decision trees use splitting criteria like Gini Impurity, Entropy, and MSE.

- They are prone to overfitting without proper tuning.

- They are interpretable and versatile, suitable for both classification and regression.

##
---